# VytalLink Time-Series Anomaly Detection

This notebook demonstrates how to load historical health data extracted from the VytalLink API, visualize resting heart rate trends, and train a basic LSTM (Long Short-Term Memory) neural network to forecast the next day's heart rate or detect anomalies based on prediction errors.

## 1. Extract Data (ETL)
First, we need to extract a historical window of data so we aren't querying the API continuously. The cell below runs the `etl` command included in this Health Kit to download the last 60 days of data and save it to `health_data.csv`.

> **Note:** if you already ran the ETL command in your terminal, you can skip this cell.

In [ ]:
import os

if not os.path.exists('health_data.csv'):
    !uv run vytallink-health-kit etl --days 60 --output-file health_data.csv
else:
    print("health_data.csv already exists. Skipping extraction.")

## 2. Load and Visualize Data
Next, we use `pandas` to load the dataset and `matplotlib`/`seaborn` to visualize it.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv('health_data.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

plt.figure(figsize=(14, 6))
sns.lineplot(data=df, x='date', y='resting_hr_bpm', marker="o", color="#FF5733")
plt.title('Resting Heart Rate over Time', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Resting Heart Rate (BPM)')
plt.show()

## 3. Train an LSTM Autoencoder / Forecaster
We will use `torch` to define a simple LSTM that predicts the next value in the sequence. Large deviations (errors) between the predicted value and the actual value could indicate a potential anomaly (e.g., incomplete recovery or impending illness).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Preprocess the HR data
hr_series = df['resting_hr_bpm'].interpolate().fillna(method='bfill').values

# Normalize to [0, 1]
min_val = hr_series.min()
max_val = hr_series.max()
hr_normalized = (hr_series - min_val) / (max_val - min_val)

# Create sequences of length SEQ_LEN
SEQ_LEN = 5
sequences = []
targets = []
for i in range(len(hr_normalized) - SEQ_LEN):
    sequences.append(hr_normalized[i:i+SEQ_LEN])
    targets.append(hr_normalized[i+SEQ_LEN])
    
X = torch.tensor(sequences, dtype=torch.float32).unsqueeze(-1) # [batch, seq_len, features]
y = torch.tensor(targets, dtype=torch.float32).unsqueeze(-1)   # [batch, features]

class MetricsDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

loader = DataLoader(MetricsDataset(X, y), batch_size=8, shuffle=True)

class LSTMForecastModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.linear(out[:, -1, :])
        return out

model = LSTMForecastModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("Training the LSTM Model...")
for epoch in range(50):
    epoch_loss = 0.0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 | Loss: {epoch_loss/len(loader):.4f}")

## 4. Anomaly Detection based on Forecasting Errors
We can now predict the heart rate for each sequence and find days where the model's prediction significantly deviates from the actual value.

In [ ]:
model.eval()
with torch.no_grad():
    predictions_normalized = model(X).squeeze().numpy()
    
# Denormalize
predictions = predictions_normalized * (max_val - min_val) + min_val
actuals = np.array([t.item() for t in y]) * (max_val - min_val) + min_val

# Compute errors
errors = np.abs(predictions - actuals)
threshold = np.mean(errors) + 2 * np.std(errors)

anomalies = errors > threshold

plt.figure(figsize=(14, 6))
time_axis = df['date'].iloc[SEQ_LEN:]

plt.plot(time_axis, actuals, label="Actual RHR", marker="o")
plt.plot(time_axis, predictions, label="Predicted RHR", linestyle="--")
plt.scatter(time_axis[anomalies], actuals[anomalies], color="red", label="Anomalies", zorder=5, s=100)

plt.title("Resting HR Forecast & Anomaly Detection", fontsize=16)
plt.xlabel("Date")
plt.ylabel("Resting Heart Rate (BPM)")
plt.legend()
plt.show()

print("Anomalous Days Detected:")
anomaly_dates = time_axis[anomalies]
for date in anomaly_dates:
    print(f" - {date.date()}")